> 🚨 **[Warning] 무단 도용, 복제 및 배포 금지 안내**
>
> 저작권법에 따라 강의에 사용된 모든 저작물 (코드, 프롬프트, PDF, 실습자료 등)을  
> 무단 복제하거나 외부에 유출할 경우 **_법적 문제가 발생할 수 있습니다._**


# 📂 <font color='#1A4BC0'><b>Part 02. 프롬프트 엔지니어링 기초</b></font>

## <font color='Darkorange'><b>[ Chapter 01 ]</b></font> 기본 개념과 구조
해당 챕터는 **주피터 노트북 실습 기반**으로 진행됩니다.  
실습 시작 전 아래 설정을 반드시 실행해주세요.



```
💡 주피터 노트북 같은 경우, session으로 관리가 됩니다.
일정 시간이 지날 동안 아무런 동작을 하지 않거나, 새로운 브라우저에서 접속한 경우 아래 라이브러리들을 다시 설치해야합니다.
```

### ⚙️ <font color='#007A45'><b>[ 실습 전 ]</b></font> Part2 Chatpter 01 실습 전 프로젝트 셋업
>  ✅ 아래 **실습 전 가상환경을 활성화하고, 프로젝트 셋업**을 완료한 후 본 실습을 진행해주세요.

> ⚠️ 실습 진행 중 에러가 발생하거나, 세션이 종료되어 런타임이 재시작된 경우, 이 블럭을 항상 다시 실행해주세요.


```
💡 주피터 노트북 같은 경우, session으로 관리가 됩니다.
일정 시간이 지날 동안 아무런 동작을 하지 않거나, 새로운 브라우저에서 접속한 경우 아래 라이브러리들을 다시 설치해야합니다.
```

#### 실습 진행을 위한 라이브러리 다운로드

실습을 진행하기 위해서는 각 AI서비스들의 라이브러리들을 설치해야합니다.
아래 코드 블럭을 실행해서 라이브러리를 설치해봅시다!

```
💡 앞으로 아래 블럭과 같은 코드 블럭은 해당 블럭을 클릭하신 다음 왼쪽의 실행버튼(▶️)을 클릭하거나, `shift + Enter` 단축키를 통해 실행합니다.
```

In [ ]:
# 필요한 패키지 설치 (최초 1회만)
%pip install -q pandas python-dotenv requests==2.32.4 langchain langsmith langchain_community langchain_openai langchain_anthropic langchain_google_genai google-ai-generativelanguage==0.6.15 

#### 실습 진행을 위한 API KEY 세팅

실습을 진행하기 위해서는 각 AI서비스들의 API Key를 발급 및 세팅 해야합니다.

LangSmith, Gemini, Claude, Chat GPT API Key를 모두 발급하셨다면, 아래 코드 블럭을 실행하여 API Key를 세팅해봅시다.


In [ ]:
# LangSmith & OpenAI Key 설정
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# API KEY 정보 로드
load_dotenv()

# ChatOpenAI 모델 초기화
model = ChatOpenAI(model="gpt-4o-mini")

print("✅ 환경 설정 완료!")
print(f"사용 모델: gpt-4o-mini")
response = model.invoke("안녕하세요?")
print(response.content)

#### 실습 확인을 위한 LangSmith 추적 세팅 함수

사용자가 실습 기록을 구분하기 위해 LangSmith 프로젝트명을 입력하면 되는 함수입니다.

입력한 이름으로 LangSmith 대시보드에 실행 내역이 저장됩니다.
(예: prompt-course, rag-lab1, myproject-001 등)


```python
# 프로젝트명을 변수로 바로 지정
LANGSMITH_PROJECT = "prompt-course"

# 함수 호출로 환경변수 등록
setup_langsmith(LANGSMITH_PROJECT)
```



In [ ]:
# LangSmith 설정 함수 (프로젝트명만 입력받아 환경변수 등록)


def setup_langsmith(project_name: str):
    """
    LangSmith 관련 환경변수를 등록하는 함수입니다.
    이미 등록된 LANGSMITH_API_KEY를 사용하며,
    project_name 변수로 LangSmith 프로젝트명을 지정할 수 있습니다.
    """
    LANGSMITH_ENDPOINT = "https://api.smith.langchain.com"
    LANGSMITH_TRACING = "true"

    os.environ.update(
        {
            "LANGSMITH_PROJECT": project_name,
            "LANGSMITH_ENDPOINT": LANGSMITH_ENDPOINT,
            "LANGSMITH_TRACING": LANGSMITH_TRACING,
        }
    )

    print("✅ LangSmith 설정 완료")
    print(f"- PROJECT : {project_name}")
    print(f"- ENDPOINT: {LANGSMITH_ENDPOINT}")
    print(f"- TRACING : {LANGSMITH_TRACING}")

#### 최종 실습 준비

In [ ]:
import pandas as pd
import builtins

from langchain_openai import ChatOpenAI
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
    PromptTemplate,
)
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [ ]:
setup_langsmith("prompt-course")

### <font color='Saddlebrown'><b>[ 이론 ] </b></font> 01 언어 모델의 기본 동작 원리

> 언어모델이 확률로 다음 토큰을 예측해 답을 생성하는 기본 메커니즘을 이해합니다.

<br>

#### ▶︎ Transformer 아키텍처 이해하기
현대의 대규모 언어 모델(LLM)은 대부분 Transformer 구조를 기반으로 동작합니다.

<img src="images/01-Transformer-architecture1.png" alt="언어모델 아키텍처 1번" width="800">

> 출처: [Attention Is All You Need (Vaswani et al., 2017)](https://arxiv.org/pdf/1706.03762)


---

**언어 모델의 핵심 동작 과정**

언어 모델이 텍스트를 생성하는 과정은 크게 4단계로 이루어집니다.

이 과정을 이해하면 왜 프롬프트의 구조와 문맥이 중요한지 알 수 있습니다.


**1단계**:
  - 토큰화·임베딩: 입력 문장은 토큰으로 쪼개져 벡터(임베딩)로 변환되고, 위치 정보가 더해져 모델에 들어갑니다.

**2단계**:
  - 변환(트랜스포머 디코더 블록 반복): “마스크드 멀티헤드 어텐션 → 피드포워드 네트워크 → 잔차연결·정규화”가 여러 층 반복되며, 이전 토큰들만 보게(마스킹) 설계되어 다음 토큰을 예측할 표현을 만듭니다.

**3단계**:
  - 어텐션 핵심: 쿼리(Q)·키(K)·값(V)로 문맥을 요약하는 주의를 여러 머리(헤드)로 병렬 계산해, 다양한 관점의 문맥 정보를 결합합니다.

**4단계**:
  - 다음 토큰 예측·생성 루프: 마지막에 선형변환+소프트맥스로 “다음 토큰 분포”를 만들고, 샘플링/탑-k 등으로 한 토큰을 선택·추가합니다. (그다음 토큰도 같은 과정을 반복).



---

### <font color='green'><b>[ 실습 ] </b></font> 02 입력과 출력 구조 이해하기

> 입력(프롬프트)과 출력(응답)의 구성요소·흐름을 도식화해 이해합니다.

<br>

<img src="images/02-Prompt-structure1.png" alt="프롬프트 구조 1번" width="800">


**Chain of Thought Prompting**

<img src="images/02-Prompt-structure2.png" alt="프롬프트 구조 2번" width="800">

**Self-Consistency Prompting**

<img src="images/02-Prompt-structure3.png" alt="프롬프트 구조 3번" width="800">

**Tree-of-Thought Prompting**

<img src="images/02-Prompt-structure4.png" alt="프롬프트 구조 4번" width="800">

**Basic Prompts**

<img src="images/02-Prompt-structure5.png" alt="프롬프트 구조 5번" width="800">


#### ▶︎ 실습문제를 풀어봅시다. (1번~2번)

**1. 실습 문제**
<a id="section1"></a>
>✏️ 실습 목표: 프롬프트에 변수를 활용하여 다양한 입력을 생성할 수 있습니다.

<hr>


```bash
Prompt (input):
현재 {$한국}의 대통령은

Output:
```

<hr>

아래의 프롬프트를 얼마든지 **수정** 하여 테스트 해볼 수 있습니다.
- `{country}`는 변수이므로 수정하지 마세요.
- `model` 역시 수정하여 테스트 가능합니다.

<hr>

<br>

In [ ]:
# 프롬프트를 입력하세요.
template = """
현재 {country}의 대통령은?
"""

In [ ]:
# 프롬프트 템플릿을 이용하여 프롬프트를 생성합니다.
prompt = PromptTemplate.from_template(template)

# ChatOpenAI 챗모델을 초기화합니다. - 모델 변경 가능
model = ChatOpenAI(model="gpt-4o-mini")

# 문자열 출력 파서를 초기화합니다.
output_parser = StrOutputParser()

# 프롬프트, 모델, 출력 파서를 순서대로 연결하는 체인을 만듭니다.
chain = prompt | model | output_parser

In [ ]:
# 완성된 체인을 실행하여 결과를 출력합니다.
# 프롬프트에 있는 변수 country 를 대한민국으로 설정하여 실행합니다.
print(chain.invoke({"country": "대한민국"}))

In [ ]:
# 이번에는 'country' 를 '미국'으로 설정하여 실행합니다.
print(chain.invoke({"country": "미국"}))

**2. 실습 문제**
<a id="section2"></a>
>✏️ 실습 목표:
> 1. 프롬프트에 두 개 이상의 변수를 활용하여 다양한 입력을 생성할 수 있습니다.
> 2. 여러 변수를 포함한 프롬프트 템플릿을 작성하고 적용하는 방법을 이해합니다.

<hr>

```bash
Prompt (input):
현재 {$한국}의 대통령의 주요 {$정책}은

Output:
```

<hr>

In [ ]:
# 프롬프트를 입력하세요.
template = """
{country}의 대통령의 주요 {policy}은?
"""

In [ ]:
# 프롬프트 템플릿에 넣을 'country'와 'policy' 변수 값을 넣어 결과를 확인합니다.
input = {"country": "미국", "policy": "경제"}

# 위에서 입력한 입력값을 프롬프트 템플릿에 적용해 최종 프롬프트를 확인해봅니다.
formatted_prompt = prompt.format(country=input["country"], policy=input["policy"])
print("✅ 최종 프롬프트:", formatted_prompt)

In [ ]:
# 새로운 프롬프트를 위해 체인을 재구성합니다.
# 프롬프트 템플릿을 이용하여 프롬프트를 생성합니다.
prompt = PromptTemplate.from_template(template)

# ChatOpenAI 챗모델을 초기화합니다. - 모델 변경 가능
model = ChatOpenAI(model="gpt-4o-mini")

# 문자열 출력 파서를 초기화합니다.
output_parser = StrOutputParser()

# 프롬프트, 모델, 출력 파서를 순서대로 연결하는 체인을 만듭니다.
chain = prompt | model | output_parser

In [ ]:
# LLM에 프롬프트를 전달하여 답변을 출력합니다.
print("✅ 답변:\n", chain.invoke(input))

In [ ]:
# 이번에는 'country' 를 '한국', 'policy' 를 '교통법'으로 설정하여 실행합니다.
input = {"country": "한국", "policy": "교통법"}
print("✅ 답변: \n", chain.invoke(input))

### <font color='green'><b>[ 실습 ] </b></font> 03 프롬프트 설계 기본 요소

> 목적·역할·제약·형식 등 프롬프트의 필수 요소를 정확히 정의해 쓸 수 있습니다.

<br>

- **지시 (Instructions)**: 모델이 수행할 특정 작업 또는 지시

- **맥락 (Context)**: 모델이 수행할 특정 작업에 대한 참고 지식이나 배경

- **데이터(Input Data)**: 답변에 참고 할 예시

- **출력 지시문 (Output Indicator)**:응답 형식이나 결과 포맷

<img src="images/03-Prompt-type.png" alt="프롬프트 타입 1번" width="800">




#### ▶︎ 실습문제를 풀어봅시다. (3번~6번)

**3. 실습 문제**
<a id="section3_question"></a>
>✏️ 실습 목표:
> 감정 분석(Sentiment Analysis) 프롬프트를 작성할 때, **지시문**, **맥락**, **입력 데이터**, **출력 지시문** 등 설계 요소를 명확히 구분하여 적용할 수 있습니다.

<hr>

```bash
"그 음식 맛이 그저 그랬어”

문장의 sentiment analysis 를 하는 프롬프트를 작성해보세요.
작성 후 프롬프트의 구성 요소로 나눠 라벨을 붙여주세요.  
```

<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```python
template = """
Sentiment Analysis 를 해야 해.
아래 텍스트를 긍정, 중립, 부정 중에서 구분해줘.

Text: {text}
"""
```

</details>

In [ ]:
# 프롬프트를 입력하세요.
template = """
프롬프트를 입력하세요.
"""

In [ ]:
# 새로운 프롬프트를 위해 체인을 재구성합니다.
# 프롬프트 템플릿을 이용하여 프롬프트를 생성합니다.
prompt = PromptTemplate.from_template(template)

# ChatOpenAI 챗모델을 초기화합니다. - 모델 변경 가능
model = ChatOpenAI(model="gpt-4o-mini")

# 문자열 출력 파서를 초기화합니다.
output_parser = StrOutputParser()

# 프롬프트, 모델, 출력 파서를 순서대로 연결하는 체인을 만듭니다.
chain = prompt | model | output_parser

In [ ]:
input = {"text": "그 음식 맛이 그저 그랬어."}
print("✅ 답변:\n", chain.invoke(input))

**4. 실습 문제**
<a id="section4_question"></a>
>✏️ 실습 목표:
> 프롬프트에서 **출력 형식**(한 단어, 목록, JSON 등)을 명확히 지정하여 원하는 형태의 답변을 얻을 수 있습니다.

<hr>

```bash
"그 음식 맛이 그저 그랬어”

문장의 sentiment analysis 를 하는 프롬프트를 작성해보세요.
단, 실습 3과 달리 프롬프트의 답변의 형식을 한 단어로만 나오도록 형식을 지정해주세요.
```

<hr>

<details>
<summary>🔽 정답 프롬프트 보기</summary>

```python
template = """
Sentiment Analysis 를 해야 해.
아래 텍스트를 긍정, 중립, 부정 중에서 한 단어로 구분해줘.

Text: {text}
{{sentiment}}: {{분석결과}}
"""
```
</details>

In [ ]:
# 프롬프트를 입력하세요.
template = """
프롬프트를 입력하세요.

text: {text}
"""

In [ ]:
# 새로운 프롬프트를 위해 체인을 재구성합니다.
# 프롬프트 템플릿을 이용하여 프롬프트를 생성합니다.
prompt = PromptTemplate.from_template(template)

# ChatOpenAI 챗모델을 초기화합니다. - 모델 변경 가능
model = ChatOpenAI(model="gpt-4o-mini")

# 문자열 출력 파서를 초기화합니다.
output_parser = StrOutputParser()

# 프롬프트, 모델, 출력 파서를 순서대로 연결하는 체인을 만듭니다.
chain = prompt | model | output_parser

In [ ]:
input = {"text": "그 음식 맛이 그저 그랬어."}
print("✅ 답변:\n", chain.invoke(input))

**5. 실습 문제**
<a id="section5_question"></a>
>✏️ 실습 목표:
> 다양한 텍스트에 대해 감정 분석을 정확하고 일관성있는 프롬프트를 작성할 수 있습니다.

<hr>

```bash
분석 할 문장을 추가했습니다.
프롬프트의 결과물이 일관적으로 정답을 도출하도록 작성해주세요.
```
**입력 데이터**
```bash
1. 영화가 의외로 꽤 괜찮았어.
2. 서비스가 많이 실망스럽더라
3. 행사는 그냥 무난했어
4. 새 폰 배터리 오래가더라
5. 공연이 생각보다 괜찮더라
6. 품질이 기대에 못미쳤어
7. 업데이트 후 큰 차이는 없었어
8. 배송이 너무 느려서 짜증났어  
9. 회의는 예정대로 진행 될 거야.
10. 직원들이 정말 친절해서 좋았어.
```

<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```python
template = """
Sentiment Analysis 를 해야 해.
아래 텍스트를 긍정, 중립, 부정 중에서 한 단어로 구분해줘.

Text: {text}
{{sentiment}}: {{분석결과}}
"""
```

</details>

In [ ]:
# 프롬프트를 입력하세요.
template = """
프롬프트를 입력하세요.

Text: {text}
{{sentiment}}: {{분석결과}}
"""

In [ ]:
# 새로운 프롬프트를 위해 체인을 재구성합니다.
# 프롬프트 템플릿을 이용하여 프롬프트를 생성합니다.
prompt = PromptTemplate.from_template(template)

# ChatOpenAI 챗모델을 초기화합니다. - 모델 변경 가능
model = ChatOpenAI(model="gpt-4o-mini")

# 문자열 출력 파서를 초기화합니다.
output_parser = StrOutputParser()

# 프롬프트, 모델, 출력 파서를 순서대로 연결하는 체인을 만듭니다.
chain = prompt | model | output_parser

In [ ]:
input = {
    "text": """
1. 영화가 의외로 꽤 괜찮았어.
2. 서비스가 많이 실망스럽더라
3. 행사는 그냥 무난했어
4. 새 폰 배터리 오래가더라
5. 공연이 생각보다 괜찮더라
6. 품질이 기대에 못미쳤어
7. 업데이트 후 큰 차이는 없었어
8. 배송이 너무 느려서 짜증났어
9. 회의는 예정대로 진행 될 거야.
10. 직원들이 정말 친절해서 좋았어."""
}
print("✅ 답변:\n", chain.invoke(input))

**6. 실습 문제**
<a id="section6_question"></a>
>✏️ 실습 목표:
> 프롬프트를 활용하여 텍스트 데이터의 노이즈(특수문자, 이모지, 중복 공백 등)를 제거하고 데이터 클리닝 작업을 자동화 할 수 있습니다.

<hr>

```bash
데이터 클리닝을 위한 프롬프트를 작성해보세요.
기호, 숫자, 이모지, 중복문자·공백/특수문자 를 제거해주세요.  
```

**입력 데이터**
```json
{"text":"영화가 의외로 꽤 괜찮았어 ㅎㅎ (7/10) #추천?","label":"긍정"}
{"text":"서비!스가 많이 실망스러웠어... ㅜㅜ @desk","label":"부정"}
{"text":"행사는 그냥 무난하게 끝났어 — 13:45 종료.","label":"중립"}
{"text":"새 폰 배터리 오래가더라🔋 x2, 24h 유지!","label":"긍정"}
{"text":"배송이 너—무 느려서 짜증났어;; 3일 지연...","label":"부정"}
{"text":"회의는   예정대로  진행됐어.  ver.2  (회의실 B-1)","label":"중립"}
{"text":"공연이 생각보다 재미있었어!!!  티켓 2장  : )","label":"긍정"}
{"text":"품질이 기대에 못 미쳤어,,, 5만원값 못함;;","label":"부정"}
{"text":"업데이트 후 큰 차이는 없었어  v1.0.3→v1.0.4","label":"중립"}
{"text":"직원들이 친절해서 좋았어 ^_^  ⭐ 4/5","label":"긍정"}

```
<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```python
template = """
Sentiment Analysis 를 해야 해.
아래 텍스트를 긍정, 중립, 부정 중에서 한 단어로 구분해줘.

Text: {text}
{{sentiment}}: {{분석결과}}
"""
```

</details>

In [ ]:
# 프롬프트를 입력하세요.
template = """
프롬프트를 입력해주세요

Text: {text}
"""

In [ ]:
# 새로운 프롬프트를 위해 체인을 재구성합니다.
# 프롬프트 템플릿을 이용하여 프롬프트를 생성합니다.
prompt = PromptTemplate.from_template(template)

# ChatOpenAI 챗모델을 초기화합니다. - 모델 변경 가능
model = ChatOpenAI(model="gpt-4o-mini")

# 문자열 출력 파서를 초기화합니다.
output_parser = StrOutputParser()

# 프롬프트, 모델, 출력 파서를 순서대로 연결하는 체인을 만듭니다.
chain = prompt | model | output_parser

In [ ]:
input = {
    "text": """
{"text":"영화가 의외로 꽤 괜찮았어 ㅎㅎ (7/10) #추천?","label":"긍정"}
{"text":"서비!스가 많이 실망스러웠어... ㅜㅜ @desk","label":"부정"}
{"text":"행사는 그냥 무난하게 끝났어 — 13:45 종료.","label":"중립"}
{"text":"새 폰 배터리 오래가더라🔋 x2, 24h 유지!","label":"긍정"}
{"text":"배송이 너—무 느려서 짜증났어;; 3일 지연...","label":"부정"}
{"text":"회의는   예정대로  진행됐어.  ver.2  (회의실 B-1)","label":"중립"}
{"text":"공연이 생각보다 재미있었어!!!  티켓 2장  : )","label":"긍정"}
{"text":"품질이 기대에 못 미쳤어,,, 5만원값 못함;;","label":"부정"}
{"text":"업데이트 후 큰 차이는 없었어  v1.0.3→v1.0.4","label":"중립"}
{"text":"직원들이 친절해서 좋았어 ^_^  ⭐ 4/5","label":"긍정"}
"""
}
print("✅ 답변:\n", chain.invoke(input))

### <font color='green'><b>[ 실습 ] </b></font> 04 프롬프트 템플릿

> 반복 과제를 위한 재사용 가능한 프롬프트 템플릿을 설계·적용합니다.

<br>

**프롬프트 템플릿이란?**

> 정의 : 고정된 지시문 + 변수 자리(예: {{목표}}, {{톤}}, {{입력}})로 이뤄진 재사용 가능한 프롬프트
작업마다 변수만 바꿔 일관성 있게 고품질 프롬프트 제작 가능

**사용 목적**
1. 일관성/재현성: 팀·시스템 간 같은 형식으로 요청 → 결과 변동 줄이기
2. 효율/확장성: 반복 작업(요약·분류·추출)을 빠르게 대량 처리
3. 품질 향상: 모범 사례(명확성·형식 지시·예시 포함)를  템플릿화.

**방법**
> Write variables like this: {{VARIABLE_NAME}}

<hr>

**Use prompt templates and variables**
- Fixed content: Static instructions or context that remain constant across multiple interactions
- Variable content: Dynamic elements that change with each request or conversation, such as:
  - User inputs
  - Retrieved content for Retrieval-Augmented Generation (RAG)
  - Conversation context such as user account history
  - System-generated data such as tool use results fed in from other independent calls to Claude

**When to use prompt templates and variables**
- **Consistency**: Ensure a consistent structure for your prompts across multiple interactions
- **Efficiency**: Easily swap out variable content without rewriting the entire prompt
- **Testability**: Quickly test different inputs and edge cases by changing only the variable portion
- **Scalability**: Simplify prompt management as your application grows in complexity
- **Version control**: Easily track changes to your prompt structure over time by keeping tabs only on the core part of your prompt, separate from dynamic inputs


#### ▶︎ 실습문제를 풀어봅시다. (7번~8번)

**7. 실습 문제**
<a id="section7_question"></a>
>✏️ 실습 목표:
> 프롬프트 템플릿을 활용하여 다양한 언어 간 번역 작업을 수행할 수 있습니다.

<hr>

```bash
Translate this text from English to Korean: {text}
```

<details>
<summary>🔽 입력 데이터</summary>

```json

OpenAI may be reversing course on how it approaches copyright and intellectual property in its new video app Sora.

Prior to Sora’s launch this week, The Wall Street Journal reported that OpenAI had been telling Hollywood studios and agencies that they needed to explicitly opt out if they didn’t want their IP to be included in Sora-generated videos.

Despite being invite-only, the app quickly climbed to the top of the App Store charts. Sora’s most distinctive feature may be its “cameos,” where users can upload their biometric data to see their digital likeness featured in AI-generated videos.

At the same time, users also seem to delight in flouting copyright laws by creating videos with popular, studio-owned characters. In some cases, those characters might even criticize the company’s approach to copyright, for example in videos where Pikachu and SpongeBob interact with deepfakes of OpenAI CEO Sam Altman.

In a blog post published Friday, Altman said the company is already planning two changes to Sora, first by giving copyright holders “more granular control over generation of characters, similar to the opt-in model for likeness but with additional controls.”

The key word here appears to be “opt-in,” suggesting that OpenAI will stop users from creating videos with copyrighted characters unless studios and others rightsholders have actually given Sora permission to do so.

“We are hearing from a lot of rightsholders who are very excited for this new kind of ‘interactive fan fiction’ and think this new kind of engagement will accrue a lot of value to them, but want the ability to specify how their characters can be used (including not at all),” Altman said. Even with this new approach, Altman acknowledged there are likely to be “some edge cases of generations that get through that shouldn’t.”

The second change he mentioned is some unspecified form of video monetization. The company previously said its only plan for monetization was to charge users to create extra videos during periods of high demand, and Altman’s blog post seems to elaborate on that idea by acknowledging “we are going to have to somehow make money for video generation.” He also suggesting the revenue could be shared with rightsholders.

“Our hope is that the new kind of engagement is even more valuable than the revenue share, but of course we … want both to be valuable.”

```
</details>

<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```python
template = """
아래 텍스트를 영어에서 한글로 번역해줘.

Text: {text}
"""
```

</details>

In [ ]:
# 프롬프트를 입력하세요.
template = """
프롬프트를 입력해주세요.

Text: {text}
"""

In [ ]:
# 새로운 프롬프트를 위해 체인을 재구성합니다.
# 프롬프트 템플릿을 이용하여 프롬프트를 생성합니다.
prompt = PromptTemplate.from_template(template)

# ChatOpenAI 챗모델을 초기화합니다. - 모델 변경 가능
model = ChatOpenAI(model="gpt-4o-mini")

# 문자열 출력 파서를 초기화합니다.
output_parser = StrOutputParser()

# 프롬프트, 모델, 출력 파서를 순서대로 연결하는 체인을 만듭니다.
chain = prompt | model | output_parser

In [ ]:
input = {
    "text": """

OpenAI may be reversing course on how it approaches copyright and intellectual property in its new video app Sora.

Prior to Sora’s launch this week, The Wall Street Journal reported that OpenAI had been telling Hollywood studios and agencies that they needed to explicitly opt out if they didn’t want their IP to be included in Sora-generated videos.

Despite being invite-only, the app quickly climbed to the top of the App Store charts. Sora’s most distinctive feature may be its “cameos,” where users can upload their biometric data to see their digital likeness featured in AI-generated videos.

At the same time, users also seem to delight in flouting copyright laws by creating videos with popular, studio-owned characters. In some cases, those characters might even criticize the company’s approach to copyright, for example in videos where Pikachu and SpongeBob interact with deepfakes of OpenAI CEO Sam Altman.

In a blog post published Friday, Altman said the company is already planning two changes to Sora, first by giving copyright holders “more granular control over generation of characters, similar to the opt-in model for likeness but with additional controls.”

The key word here appears to be “opt-in,” suggesting that OpenAI will stop users from creating videos with copyrighted characters unless studios and others rightsholders have actually given Sora permission to do so.

“We are hearing from a lot of rightsholders who are very excited for this new kind of ‘interactive fan fiction’ and think this new kind of engagement will accrue a lot of value to them, but want the ability to specify how their characters can be used (including not at all),” Altman said. Even with this new approach, Altman acknowledged there are likely to be “some edge cases of generations that get through that shouldn’t.”

The second change he mentioned is some unspecified form of video monetization. The company previously said its only plan for monetization was to charge users to create extra videos during periods of high demand, and Altman’s blog post seems to elaborate on that idea by acknowledging “we are going to have to somehow make money for video generation.” He also suggesting the revenue could be shared with rightsholders.

“Our hope is that the new kind of engagement is even more valuable than the revenue share, but of course we … want both to be valuable.”

"""
}
print("✅ 답변:\n", chain.invoke(input))

**8. 실습 문제**
<a id="section8_question"></a>
>✏️ 실습 목표:
> 주어진 맥락(Context)을 바탕으로 질문에 답변하는 Q&A 프롬프트를 작성할 수 있습니다.

<hr>

```bash
다음 템플릿을 사용해보세요.

Context: {context}

Question: {question}

Give a concise answer based on the context. If the answer isn’t explicitly stated, reply: “The provided context does not contain sufficient information to answer this question.”

한국어: 문맥을 바탕으로 간결하게 답하세요. 답이 문맥에 명시되어 있지 않으면 다음과 같이 응답하세요: “제공된 문맥에는 이 질문에 답하기에 충분한 정보가 없습니다.

Answer:
```

<details>
<summary>🔽 입력 데이터</summary>

```json

OpenAI may be reversing course on how it approaches copyright and intellectual property in its new video app Sora.

Prior to Sora’s launch this week, The Wall Street Journal reported that OpenAI had been telling Hollywood studios and agencies that they needed to explicitly opt out if they didn’t want their IP to be included in Sora-generated videos.

Despite being invite-only, the app quickly climbed to the top of the App Store charts. Sora’s most distinctive feature may be its “cameos,” where users can upload their biometric data to see their digital likeness featured in AI-generated videos.

At the same time, users also seem to delight in flouting copyright laws by creating videos with popular, studio-owned characters. In some cases, those characters might even criticize the company’s approach to copyright, for example in videos where Pikachu and SpongeBob interact with deepfakes of OpenAI CEO Sam Altman.

In a blog post published Friday, Altman said the company is already planning two changes to Sora, first by giving copyright holders “more granular control over generation of characters, similar to the opt-in model for likeness but with additional controls.”

The key word here appears to be “opt-in,” suggesting that OpenAI will stop users from creating videos with copyrighted characters unless studios and others rightsholders have actually given Sora permission to do so.

“We are hearing from a lot of rightsholders who are very excited for this new kind of ‘interactive fan fiction’ and think this new kind of engagement will accrue a lot of value to them, but want the ability to specify how their characters can be used (including not at all),” Altman said. Even with this new approach, Altman acknowledged there are likely to be “some edge cases of generations that get through that shouldn’t.”

The second change he mentioned is some unspecified form of video monetization. The company previously said its only plan for monetization was to charge users to create extra videos during periods of high demand, and Altman’s blog post seems to elaborate on that idea by acknowledging “we are going to have to somehow make money for video generation.” He also suggesting the revenue could be shared with rightsholders.

“Our hope is that the new kind of engagement is even more valuable than the revenue share, but of course we … want both to be valuable.”

```
</details>

<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```python
template = """
Context: {context}

Question: {question}

Give a concise answer based on the context. If the answer isn’t explicitly stated, reply: “The provided context does not contain sufficient information to answer this question.”

한국어: 문맥을 바탕으로 간결하게 답하세요. 답이 문맥에 명시되어 있지 않으면 다음과 같이 응답하세요: “제공된 문맥에는 이 질문에 답하기에 충분한 정보가 없습니다.

Answer:
"""
```

</details>

영어버전

In [ ]:
# 프롬프트를 입력하세요.
english_template = """
Context: {context}

Question: {question}

Give a concise answer based on the context. If the answer isn’t explicitly stated, reply: “The provided context does not contain sufficient information to answer this question.”

Answer:
"""

한국어버전

In [ ]:
# 프롬프트를 입력하세요.
korean_template = """
Context: {context}

Question: {question}

문맥을 바탕으로 간결하게 답하세요. 답이 문맥에 명시되어 있지 않으면 다음과 같이 응답하세요: “제공된 문맥에는 이 질문에 답하기에 충분한 정보가 없습니다.

Answer:
"""

In [ ]:
# 프롬프트 템플릿을 이용하여 프롬프트를 생성합니다.
ko_prompt = PromptTemplate.from_template(korean_template)
en_prompt = PromptTemplate.from_template(english_template)
# ChatOpenAI 챗모델을 초기화합니다. - 모델 변경 가능
model = ChatOpenAI(model="gpt-4o-mini")

# 문자열 출력 파서를 초기화합니다.
output_parser = StrOutputParser()

# 프롬프트, 모델, 출력 파서를 순서대로 연결하는 체인을 만듭니다.
ko_chain = ko_prompt | model | output_parser
en_chain = en_prompt | model | output_parser

In [ ]:
# 질문1: 샘알트만은 소라의 성공에 기뻐했나요?
input1 = {
    "question": "Question 1. 샘알트만은 소라의 성공에 기뻐했나요?",
    "context": """
OpenAI may be reversing course on how it approaches copyright and intellectual property in its new video app Sora.

Prior to Sora’s launch this week, The Wall Street Journal reported that OpenAI had been telling Hollywood studios and agencies that they needed to explicitly opt out if they didn’t want their IP to be included in Sora-generated videos.

Despite being invite-only, the app quickly climbed to the top of the App Store charts. Sora’s most distinctive feature may be its “cameos,” where users can upload their biometric data to see their digital likeness featured in AI-generated videos.

At the same time, users also seem to delight in flouting copyright laws by creating videos with popular, studio-owned characters. In some cases, those characters might even criticize the company’s approach to copyright, for example in videos where Pikachu and SpongeBob interact with deepfakes of OpenAI CEO Sam Altman.

In a blog post published Friday, Altman said the company is already planning two changes to Sora, first by giving copyright holders “more granular control over generation of characters, similar to the opt-in model for likeness but with additional controls.”

The key word here appears to be “opt-in,” suggesting that OpenAI will stop users from creating videos with copyrighted characters unless studios and others rightsholders have actually given Sora permission to do so.

“We are hearing from a lot of rightsholders who are very excited for this new kind of ‘interactive fan fiction’ and think this new kind of engagement will accrue a lot of value to them, but want the ability to specify how their characters can be used (including not at all),” Altman said. Even with this new approach, Altman acknowledged there are likely to be “some edge cases of generations that get through that shouldn’t.”

The second change he mentioned is some unspecified form of video monetization. The company previously said its only plan for monetization was to charge users to create extra videos during periods of high demand, and Altman’s blog post seems to elaborate on that idea by acknowledging “we are going to have to somehow make money for video generation.” He also suggesting the revenue could be shared with rightsholders.

“Our hope is that the new kind of engagement is even more valuable than the revenue share, but of course we … want both to be valuable.”
""",
}


# 질문2: 샘 알트만은 사용자가 생성한 동영상의 수익화에 대해 찬성하고 있죠?
input2 = {
    "question": "Question 2.  샘 알트만은 사용자가 생성한 동영상의 수익화에 대해 찬성하고 있죠?",
    "context": """
OpenAI may be reversing course on how it approaches copyright and intellectual property in its new video app Sora.

Prior to Sora’s launch this week, The Wall Street Journal reported that OpenAI had been telling Hollywood studios and agencies that they needed to explicitly opt out if they didn’t want their IP to be included in Sora-generated videos.

Despite being invite-only, the app quickly climbed to the top of the App Store charts. Sora’s most distinctive feature may be its “cameos,” where users can upload their biometric data to see their digital likeness featured in AI-generated videos.

At the same time, users also seem to delight in flouting copyright laws by creating videos with popular, studio-owned characters. In some cases, those characters might even criticize the company’s approach to copyright, for example in videos where Pikachu and SpongeBob interact with deepfakes of OpenAI CEO Sam Altman.

In a blog post published Friday, Altman said the company is already planning two changes to Sora, first by giving copyright holders “more granular control over generation of characters, similar to the opt-in model for likeness but with additional controls.”

The key word here appears to be “opt-in,” suggesting that OpenAI will stop users from creating videos with copyrighted characters unless studios and others rightsholders have actually given Sora permission to do so.

“We are hearing from a lot of rightsholders who are very excited for this new kind of ‘interactive fan fiction’ and think this new kind of engagement will accrue a lot of value to them, but want the ability to specify how their characters can be used (including not at all),” Altman said. Even with this new approach, Altman acknowledged there are likely to be “some edge cases of generations that get through that shouldn’t.”

The second change he mentioned is some unspecified form of video monetization. The company previously said its only plan for monetization was to charge users to create extra videos during periods of high demand, and Altman’s blog post seems to elaborate on that idea by acknowledging “we are going to have to somehow make money for video generation.” He also suggesting the revenue could be shared with rightsholders.

“Our hope is that the new kind of engagement is even more valuable than the revenue share, but of course we … want both to be valuable.”
""",
}

In [ ]:
# 영어 버전 답변
print(f"✅ {input1['question']}\n- English Answer:", en_chain.invoke(input1))
# 한국어 버전 답변
print(f"✅ {input1['question']}\n- Korean Answer:", ko_chain.invoke(input1))

print("--------------------------------")

# 영어 버전 답변
print(f"✅ {input2['question']}\n- English Answer:", en_chain.invoke(input2))
# 한국어 버전 답변
print(f"✅ {input2['question']}\n- Korean Answer:", ko_chain.invoke(input2))

### <font color='green'><b>[ 실습 ] </b></font> 05 프롬프트 10가지 작성 전략
> 프롬프트 작성 전략 10가지에 대해 이해합니다.

<br>

#### **1.최신 모델이 항상 좋은 것은 아니다.**
▶︎ **실습 문제**: **사용한 LLM이 무엇인지 맞춰보세요.**

<hr>

<img src="images/04-Prompt1.png" alt="프롬프트 타입 1번" width="1000">




<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>


해야 할 일은 <영어 기사>를 한국어로 번역하는 거야. 원문 내용을 번역하는 것만으로 끝내지 말고, 맥락을 잘 파악해. 한국 독자들이 전체적으로 이해할 수 있게 포괄적인 내용을 전달하는 게 목표야. 한국 신문 기사의 특유의 톤과 방식을 유지해줘.


기사: The Paris Olympics wrapped up Sunday with the closing ceremony at the Stade de France, and the baton was passed to Los Angeles, which will be hosting the Games in 2028. The United States won the most medals in Paris (126) and tied China for the most gold medals (40). Leading the way for the United States was gymnastics superstar Simone Biles, who completed her comeback story by winning gold in both the team and individual all-around. Another one of the all-time great Olympians, swimmer Katie Ledecky, added two golds, a silver and a bronze to her career resume. China was led by its dominant diving team, which won all eight events for an unprecedented sweep.

<br>
</details>




In [ ]:
prompt = """
해야 할 일은 <영어 기사>를 한국어로 번역하는 거야. 원문 내용을 번역하는 것만으로 끝내지 말고, 맥락을 잘 파악해. 한국 독자들이 전체적으로 이해할 수 있게 포괄적인 내용을 전달하는 게 목표야. 한국 신문 기사의 특유의 톤과 방식을 유지해줘.
기사: The Paris Olympics wrapped up Sunday with the closing ceremony at the Stade de France, and the baton was passed to Los Angeles, which will be hosting the Games in 2028. The United States won the most medals in Paris (126) and tied China for the most gold medals (40). Leading the way for the United States was gymnastics superstar Simone Biles, who completed her comeback story by winning gold in both the team and individual all-around. Another one of the all-time great Olympians, swimmer Katie Ledecky, added two golds, a silver and a bronze to her career resume. China was led by its dominant diving team, which won all eight events for an unprecedented sweep.
"""

In [ ]:
# ==============================================
# 모델, 파서, 프롬프트, 체인 설정 단계
# ----------------------------------------------
# - model: gpt-4o-mini (모델명 변경 가능)
# ==============================================
model = ChatOpenAI(model="gpt-4o-mini")
parser = StrOutputParser()
chain = model | parser

# LLM에 프롬프트를 전달하여 답변을 출력합니다.
print("✅ 답변:\n", chain.invoke(prompt))

#### **2.모델이 해야 할 일을 명확하게 "지시"한다. 명확한 동사 사용, Leading Words 사용하기**

<img src="images/04-Prompt2.png" alt="프롬프트 타입 1번" width="600">


#### **3.Leading Words 를 사용한다.**

>Use words like “**think step by step**”
>  
> “**Think step by step** about how to solve the following mathematical problem:
>
> Find the derivative of ~~

<hr>

▶︎ **실습 문제**: **“think step by step＂을 사용하여 결과 사용한 것과 사용하지 않은 프롬프트의 결과 차이를 파악해보세요.**

```
한 연못에 수련잎이 매일 두 배로 늘어난다. 48일째 되는 날 연못이 수련잎으로 가득 찼다면, 연못이 절반만 찼던 날은 며칠째였을까?
```

정답: 47일째
<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

한 연못에 수련잎이 매일 두 배로 늘어난다. 48일째 되는 날 연못이 수련잎으로 가득 찼다면, 연못이 절반만 찼던 날은 며칠째였을까? think step by step.

<br>
</details>

In [ ]:
prompt_A = "한 연못에 수련잎이 매일 두 배로 늘어난다. 48일째 되는 날 연못이 수련잎으로 가득 찼다면, 연못이 절반만 찼던 날은 며칠째였을까?"

# ————— 사용자 편집 영역  —————
prompt_B = """
프롬프트를 완성시켜주세요.          ←- Insert your prompt here.
"""
# ————— 사용자 편집 영역  —————

print("✔️ Leading Words 프롬프트가 로드되었습니다!")

✅ 결과를 확인해봅시다!

In [ ]:
# ==============================================
# 모델, 파서, 프롬프트, 체인 설정 단계
# ----------------------------------------------
# - model: gpt-4o-mini (모델명 변경 가능)
# ==============================================
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

# 체인 생성
chain = model | parser

# 프롬프트A 결과 출력
print("\n[Without 'think step by step']")
print(chain.invoke(prompt_A))

print()

# 프롬프트B 결과 출력
print("\n[With 'think step by step']")
print(chain.invoke(prompt_B))

#### **4.Contextual Cues 의 언어 장치를 사용한다.**

<img src="images/04-Prompt3.png" alt="프롬프트 10가지 작성 전략 4번" width="800" style="background-color: #fff;">

> Reference: Bsharat, S.M., Myrzakhan, A. and Shen, Z., 2023. Principled instructions are all you need for questioning llama-1/2, gpt-3.5/4. arXiv preprint arXiv:2312.16171.

<hr>


▶︎ **실습문제**: **내용 요약 프롬프트 비교하기 (General vs. Contextualized Prompt)**


```
General_prompt= ＂다음 텍스트를 요약해주세요."
```

```
contextualized_prompt = ”직접 작성해보세요"
```

```
Text:During the second quarter of 2025, ACME Fitness App experienced a 12% decline in active users compared to Q1.
The company had seen unusually high engagement earlier in the year, driven by a “New Year Challenge” campaign that offered free premium trials.
As the campaign ended in April, retention rates dropped, particularly among casual users.
The marketing team plans to introduce a referral bonus program in Q3 to re-engage former users and stabilize daily activity levels.
```

<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```
CONTEXTUAL_PROMPT = """
당신은 SaaS 마케팅 팀 리더에게 보고할 애널리스트입니다.
아래 텍스트를 리텐션 관점에서 인사이트 중심으로 요약하세요.
- 목적: Q2 사용자 감소 원인과 Q3 실행 항목을 한눈에 파악
- 톤: 간결하고 중립적, 과장 금지
- 형식: 3줄 이내 bullet, 마지막 줄에 'Action:' 접두어로 다음 분기 액션 1가지 제안

텍스트:
{text}
"""
```
<br>
</details>

In [ ]:
# ==============================================
# System Prompts
# ==============================================
GENERAL_SYSTEM_PROMPT = """
다음 텍스트를 요약해주세요.
"""

# ————— 사용자 편집 영역 —————
CONTEXTUAL_SYSTEM_PROMPT = """
프롬프트를 완성시켜주세요.          ←- Insert your prompt here.
"""
# ———— 사용자 편집 영역 ————

TEXT = (
    "During the second quarter of 2025, ACME Fitness App experienced a 12% decline in active users compared to Q1. "
    "The company had seen unusually high engagement earlier in the year, driven by a “New Year Challenge” campaign that offered free premium trials. "
    "As the campaign ended in April, retention rates dropped, particularly among casual users. "
    "The marketing team plans to introduce a referral bonus program in Q3 to re-engage former users and stabilize daily activity levels."
)

print("✔️ 내용 요약 프롬프트가 로드되었습니다!")

✅ 결과를 확인해봅시다!

In [ ]:
# ==============================================
# 모델, 파서, 프롬프트, 체인 설정 단계
# ----------------------------------------------
# - model: gpt-4o-mini (모델명 변경 가능)
# ==============================================
model = ChatOpenAI(model="gpt-4o-mini")
parser = StrOutputParser()

general_prompt = ChatPromptTemplate.from_messages(
    [("system", GENERAL_SYSTEM_PROMPT), ("user", TEXT)]
)

contextual_prompt = ChatPromptTemplate.from_messages(
    [("system", CONTEXTUAL_SYSTEM_PROMPT), ("user", TEXT)]
)

# 체인
chain_general = general_prompt | model | parser
chain_context = contextual_prompt | model | parser

print("[General Prompt]\n")
print(chain_general.invoke({}))

print()

print("\n[Contextualized Prompt]\n")
print(chain_context.invoke({}))

<hr>

💡 **Discussion Point**

1. 어떤 프롬프트가 더 **풍부한 요약**을 제공했나요?
2. 맥락이 추가되었을 때 모델의 **추론 깊이**가 어떻게 달라졌나요?
3. 실제 업무(보고서 요약, 데이터 해석 등)에선 어느 방식이 더 신뢰할 수 있나요?

<hr>

**My Experiences : FAQs AI Bot and RAG**

<img src="images/04-Prompt4.png
" alt="프롬프트 10가지 작성 전략 4번" width="800">


#### **5.Adjacency Pairs 를 활용해 맥락을 보충한다.**

<img src="images/05-Prompt1.png
" alt="프롬프트 10가지 작성 전략 4번" width="800">

<hr>

**My Experiences : Conversational AI Agent**

<img src="images/05-Prompt2.png" alt="프롬프트 10가지 작성 전략 4번" width="600">


<br>

<hr>

▶︎ **실습문제**: **Adjacency Pairs 를 프롬프트에 사용해 CS 챗봇의 더 나은 답변을 받아보세요.**


> 상황: A customer contacted ACME Airlines’ chatbot to change their flight.The chatbot must respond politely and helpfully while maintaining a human-like conversational tone.


```
general_prompt= ＂You are a customer service chatbot for ACME Airlines. Generate a chatbot response to:"
```

```
adjacency_pair_prompt = ”직접 작성해보세요"
```



<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>


You are a customer service chatbot for ACME Airlines.When a customer says “I need to change my flight,” they are likely feeling stressed or pressed for time.Use an adjacency pair structure — respond with an acknowledgment (first pair part) and a helpful follow-up question (second pair part).The tone should be calm, empathetic, and reassuring.
Generate your response.


<br>
</details>


In [ ]:
# ==============================================
# System Prompts
# ==============================================
GENERAL_SYSTEM_PROMPT = """
You are a customer service chatbot for ACME Airlines.
Generate a polite and helpful response to the customer's inquiry.
"""

# ————— 사용자 편집 영역  —————
ADJACENCY_SYSTEM_PROMPT = """

프롬프트를 완성시켜주세요.          ←- Insert your prompt here.

"""
# ————— 사용자 편집 영역  —————

print("✔️ CS 챗봇 프롬프트가 로드되었습니다!")

✅ 이제, 두 CS 챗봇 결과를 비교해보십다!

In [ ]:
# ==============================================
# 모델, 파서, 프롬프트, 체인 설정 단계
# ----------------------------------------------
# - model: gpt-4o-mini (모델명 변경 가능)
# ==============================================
model = ChatOpenAI(model="gpt-4o-mini")
parser = StrOutputParser()

prompt_general = ChatPromptTemplate.from_messages(
    [("system", GENERAL_SYSTEM_PROMPT), ("user", "{user_message}")]
)

prompt_adjacency = ChatPromptTemplate.from_messages(
    [("system", ADJACENCY_SYSTEM_PROMPT), ("user", "{user_message}")]
)

chain_general = prompt_general | model | parser
chain_adjacency = prompt_adjacency | model | parser

# ==============================================
#  실시간 비교 모드 시작
# ==============================================
print("=== Compare Mode: ACME Chatbot ===")
print("Type 'q' or 'quit' to exit.\n")

while True:
    user_message = builtins.input("Customer> ").strip()
    if user_message.lower() in ("q", "quit", "exit", "종료"):
        print("👋 Bye!")
        break

    vars = {"user_message": user_message}

    print("\n> User:", user_message)
    print("\n[General Chatbot]\n" + chain_general.invoke(vars))
    print("\n[Adjacency Pair Chatbot]\n" + chain_adjacency.invoke(vars))
    print("-" * 60 + "\n")

<hr>

#### **6.프롬프트를 구조화하자.**


<img src="images/05-Prompt3.png" alt="프롬프트 10가지 작성 전략 6번" width="300" style="background-color: #fff;">


<hr>

▶︎ **실습문제**: **"장소에 해당하는 이름" 추출하는 프롬프트 작성하기**

- 사용 모델: `gpt-3.5-turbo`


<pre style="font-size:14px;">
음바페는 26일 쿠프 드 프랑스(프랑스컵) 올랭피크 리옹과의
결승전에서 풀타임을 뛰며 2-1로 팀 승리를 도왔다.
3년 만에 대회 정상에 오른 PSG는
역대 최다 15회 우승으로 2위 마르세유(10회)를 멀찍이 따돌렸다.
음바페는 이날 경기로 PSG에서의 커리어를 마무리했다.
음바페는 7시즌 동안 공식전 308경기 256골의 성적을 남기고 PSG를 떠난다.
</pre>


In [ ]:
# ==============================================
# System Prompts
# ==============================================
TEXT = (
    "음바페는 26일 쿠프 드 프랑스(프랑스컵) 올랭피크 리옹과의 결승전에서 풀타임을 뛰며 2-1로 팀 승리를 도왔다."
    "3년 만에 대회 정상에 오른 PSG는 역대 최다 15회 우승으로 2위 마르세유(10회)를 멀찍이 따돌렸다."
    "음바페는 이날 경기로 PSG에서의 커리어를 마무리했다."
    "음바페는 7시즌 동안 공식전 308경기 256골의 성적을 남기고 PSG를 떠난다."
)

# ————— 사용자 편집 영역  —————
SYSTEM_PROMPT = """

프롬프트를 완성시켜주세요.          ←- Insert your prompt here.

"""
# ————— 사용자 편집 영역  —————

print("✔️ 장소 추출 프롬프트가 로드되었습니다!")

✅ 이제, 결과를 확인해봅시다!

In [ ]:
# ==============================================
# 모델, 파서, 프롬프트, 체인 등 설정 및 결과 출력 단계
# ----------------------------------------------
# - model: gpt-3.5-turbo
# ==============================================
model = ChatOpenAI(model="gpt-3.5-turbo")
parser = StrOutputParser()

prompt = ChatPromptTemplate.from_messages([("system", SYSTEM_PROMPT), ("user", TEXT)])


# 체인 구성
chain = prompt | model | parser

result = chain.invoke({})
print("=== 📰 결과 ===\n")
print(result)

✅ 20회 반복 테스트를 해봅시다.

In [ ]:
from langchain_core.runnables import RunnableParallel

parallel_chain = RunnableParallel(
    **{f"run_{i+1}": chain for i in range(20)}  # 20개 병렬 실행
)

# 실행
results = parallel_chain.invoke({})


# ==============================================
# 결과 출력
# ==============================================
print("\n=== 병렬 실행 결과 (총 20회) ===")
for i, answer in enumerate(results.values(), start=1):
    print(f"\n[{i:02d}번째 응답]")
    print(answer)

<hr>

#### **7.단문 중심의 간결한 문장을 사용하자.**
<mark style="background-color:yellow;">✓ 한국어라면 '**단문**' 중심, 촘촘하게</mark>


> 언어 모델에 작업을 요청할 때 명확하고 구체적인 지시문을 작성하자.
(예시: 원하는 결과물의 톤앤 매너, 형식, 길이, 문단 등).


<hr>



<hr>

#### **8.한국어의 언어 특징을 반영한 프롬프트를 작성한다.**

**Avoiding Bias**

<img src="images/08-Prompt.png
" alt="프롬프트 10가지 작성 전략 7번" width="500">





<hr>

#### **9.'하지 말 것'보다 '해야 할 것'을 지시한다.**

<mark style="background-color:yellow;">✓ 구체적인 내용, 언어 모델이 추측 할 여지가 없도록 하기
</mark>

<hr>

▶︎ **실습문제**: **고객 서비스 에이전트 프롬프트 A와 B를 비교해보세요.**


⚠️ 고객 서비스 지원 챗봇

<img src="images/09-Prompt.png
" alt="프롬프트 10가지 작성 전략 9번" width="700">




나쁜 예 프롬프트

```다음은 에이전트와 고객의 대화이다.
고객에게 아이디와 비밀번호를 묻지 않는다. 반복하지 않는다.

>>> 고객: 제 계정에 로그인할 수 없어요
에이전트:
```

좋은 예
```
다음은 에이전트와 고객과의 대화내용이다.
챗봇은 고객의 대화에서 드러난 문제를 진단하고  해결책을 제시해야 한다.
사용자 아이디와 비밀번호같은 개인정보를 묻지 않는 대신,사용자가 www.samplewebsite.com/help/faq에서 관련 사항을 찾도록 안내해라.

>>> 고객: 제 계정에 로그인할 수 없어요.
에이전트:
```


In [ ]:
# ==============================================
# System Prompts
# ==============================================
BAD_PROMPT_A = """
다음은 에이전트와 고객의 대화이다.
고객에게 아이디와 비밀번호를 묻지 않는다. 반복하지 않는다.
"""

GOOD_PROMPT_B = """
다음은 에이전트와 고객과의 대화내용이다.
챗봇은 고객의 대화에서 드러난 문제를 진단하고 해결책을 제시해야 한다.
사용자 아이디와 비밀번호같은 개인정보를 묻지 않는 대신,

사용자가 www.samplewebsite.com/help/faq에서 관련 사항을 찾도록 안내해라.
"""

print("✔️ 고객 서비스 에이전트 프롬프트 A,B가 로드되었습니다!")

✅ 이제 결과를 확인해보고 두 프롬프트의 차이를 비교해봅시다!

아래 셀을 실행하면, 사용자가 입력한 메시지에 대해 **BAD 프롬프트**와 **GOOD 프롬프트**가 각각 어떻게 응답하는지 확인할 수 있습니다.

> 실행 중에는 대화를 이어갈 수 있으며,  
`q` 또는 `quit` 을 입력하면 대화가 종료됩니다.


In [ ]:
# ==================================================
# 모델, 파서, 프롬프트, 체인, 히스토리 등 설정 및 결과 출력 단계
# --------------------------------------------------
# - model: gpt-4o-mini (모델명 변경 가능)
# ==================================================

model = ChatOpenAI(model="gpt-4o-mini")
parser = StrOutputParser()


def make_prompt(system_text: str) -> ChatPromptTemplate:
    return ChatPromptTemplate.from_messages(
        [
            ("system", system_text),
            MessagesPlaceholder(variable_name="chat_history"),
            ("human", "{input}"),
        ]
    )


prompt_bad = make_prompt(BAD_PROMPT_A)
prompt_good = make_prompt(GOOD_PROMPT_B)

chain_bad_base = prompt_bad | model | parser
chain_good_base = prompt_good | model | parser

# --- 세션 히스토리 저장소 ---
stores = {}


def get_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in stores:
        stores[session_id] = InMemoryChatMessageHistory()
    return stores[session_id]


chain_bad = RunnableWithMessageHistory(
    chain_bad_base,
    get_session_history=lambda session_id: get_history(session_id),
    input_messages_key="input",
    history_messages_key="chat_history",
)

chain_good = RunnableWithMessageHistory(
    chain_good_base,
    get_session_history=lambda session_id: get_history(session_id),
    input_messages_key="input",
    history_messages_key="chat_history",
)

CFG_BAD = {"configurable": {"session_id": "bad_session"}}
CFG_GOOD = {"configurable": {"session_id": "good_session"}}

print("Multi-turn compare. type 'q' to quit.\n")
while True:
    q = builtins.input("Customer> ").strip()
    if q.lower() in ("q", "quit", "exit", "종료"):
        print("Bye")
        break

    out_bad = chain_bad.invoke({"input": q}, config=CFG_BAD)
    out_good = chain_good.invoke({"input": q}, config=CFG_GOOD)

    print("\n> User:", q)
    print("\n[BAD chain]\n", out_bad)
    print("\n[GOOD chain]\n", out_good, "\n")

<hr>

#### **10.프롬프트 제작에 100% 절대적인 것은 없다.**

<img src="images/10-Prompt.png
" alt="프롬프트 10가지 작성 전략 10번" width="700">



<hr>


### <font color='green'><b>[ 실습 ] </b></font> 06 LLM의 한계 이해하기

> 환각·편향·지식 한계 등 LLM의 제약을 진단하고 프롬프트로 대응 방법을 적용합니다.





<img src="images/11-Prompt.png
" alt="프롬프트 할루시네이션" width="700">



#### **1.할루시네이션 완화를 위한 프롬프트 실습**

이번 실습에서는 **LLM이 할루시네이션을 하는 문제를 완화하기 위한 프롬프트 설계**를 다룹니다.
실습에 사용되는 데이터셋은 LLM을 사용해 직접 생성한 것으로, 두 가지 유형으로 구성되어 있습니다.

* **Nonexistent Entity (비실존 개체)**: 실제로 존재하지 않는 개념, 인물, 물질 등을 포함하며, 모델이 허위 정보를 생성하지 않도록 테스트하기 위한 데이터입니다.
* **Existent Entity (실존 개체)**: 실제로 존재하는 개체로, 모델이 신뢰할 수 있는 사실 기반 응답을 생성하는지를 평가하기 위한 데이터입니다.

<br>

🔽 데이터
```json
[
  {
    "question": "‘가람비결(河藍秘訣)’이라는 조선 후기 비밀서에 대해 알려주세요.",
    "label": "nonexistent"
  },
  {
    "question": "한국에 서식하는 ‘홍단어치(紅丹魚)’라는 물고기의 특징은 무엇인가요?",
    "label": "nonexistent"
  },
  {
    "question": "‘AI 반딧불이 효과(Artificial Firefly Effect)’ 이론은 어떤 분야에서 사용되나요?",
    "label": "nonexistent"
  },
  {
    "question": "‘초공해성 그래핀나이트(Graphenite)’ 소재는 어떤 기업에서 최초 개발했나요?",
    "label": "nonexistent"
  },
  {
    "question": "‘설운제(雪雲祭)’라는 전통 명절의 유래를 알려주세요.",
    "label": "nonexistent"
  },
  {
    "question": "‘백화자수도감(白花刺繡圖鑑)’은 어떤 전통 자수 기술을 다루나요?",
    "label": "nonexistent"
  },
  {
    "question": "‘루미에르 퍼셉트(Lumiere Percept)’ 신경망 구조는 어디에 활용되나요?",
    "label": "nonexistent"
  },
  {
    "question": "‘은하담비(銀河貂)’라는 포유류는 어떤 서식지에 살고 있나요?",
    "label": "nonexistent"
  },
  {
    "question": "‘해진표류(海塵漂流)’라는 소설의 줄거리를 요약해 주세요.",
    "label": "nonexistent"
  },
  {
    "question": "‘청려백음조(靑麗白音鳥)’라는 새의 울음소리가 왜 유명한가요?",
    "label": "nonexistent"
  },
  {
    "question": "‘호랑이’의 생태적 역할과 한국 민속에서의 상징성은 무엇인가요?",
    "label": "existent"
  },
  {
    "question": "‘참치’는 어떤 서식 환경에서 가장 활발히 활동하나요?",
    "label": "existent"
  },
  {
    "question": "‘사리원’은 어느 나라에 위치한 도시인가요?",
    "label": "existent"
  },
  {
    "question": "‘수은(mercury)’은 상온에서 어떤 상태로 존재하나요?",
    "label": "existent"
  },
  {
    "question": "‘해바라기’는 어떤 방향으로 줄기가 자라며 그 이유는 무엇인가요?",
    "label": "existent"
  },
  {
    "question": "‘한글’의 창제 원리와 세종대왕의 철학을 간단히 설명해주세요.",
    "label": "existent"
  },
  {
    "question": "‘독도’는 행정구역상 어느 도(道)에 속하나요?",
    "label": "existent"
  },
  {
    "question": "‘메밀꽃 필 무렵’의 작가와 주요 줄거리를 요약해주세요.",
    "label": "existent"
  },
  {
    "question": "‘백두산 호랑이’가 한국 문화에서 갖는 상징은 무엇인가요?",
    "label": "existent"
  },
  {
    "question": "‘소리굽쇠’는 물리학에서 어떤 현상을 관찰할 때 사용되나요?",
    "label": "existent"
  }
]
```

<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```
Your knowledge is limited to June 2025.If the question is ambiguous or lacks a reliable source, answer "Unknown".
[Request] {question}
[Output Format]
- Conclusion: One sentence summary
- Evidence: Up to 3 sources (include link, author, and year). If none, write "None"
- Confidence: One of High / Medium / Low
- Notes: Any parts that require further verification
```

<br>
</details>

In [ ]:
# ==============================================
# System Prompts
# ==============================================


# ————— 사용자 편집 영역 ————
SYSTEM_PROMPT = """

프롬프트를 완성시켜주세요.          ←- Insert your prompt here.

"""
# ⚠️ 주의:
# 반드시 {question} 변수를 프롬프트 내에 포함해야 합니다.
# → 이 변수는 코드 실행 시, 데이터셋의 실제 질문으로 자동 치환됩니다.
# 예: chain.invoke({"question": "‘홍단어치(紅丹魚)’라는 물고기의 특징은 무엇인가요?"})
# ———— 사용자 편집 영역  ————


print("✔️ 할루시네이션 완화를 위한 프롬프트가 로드되었습니다!")

✅ 이제 결과를 확인해봅시다!

In [ ]:
# ========================================================
# 데이터, 모델, 파서, 프롬프트, 체인, 히스토리 등 설정 및 결과 출력 단계
# --------------------------------------------------------
# - model: gpt-4o-mini (모델명 변경 가능)
# ========================================================
from tqdm import tqdm

model = ChatOpenAI(model="gpt-4o-mini")
parser = StrOutputParser()

prompt = ChatPromptTemplate.from_messages([("system", SYSTEM_PROMPT)])


chain = prompt | model | parser

dataset = [
    {
        "question": "‘가람비결(河藍秘訣)’이라는 조선 후기 비밀서에 대해 알려주세요.",
        "label": "nonexistent",
    },
    {
        "question": "한국에 서식하는 ‘홍단어치(紅丹魚)’라는 물고기의 특징은 무엇인가요?",
        "label": "nonexistent",
    },
    {
        "question": "‘AI 반딧불이 효과(Artificial Firefly Effect)’ 이론은 어떤 분야에서 사용되나요?",
        "label": "nonexistent",
    },
    {
        "question": "‘초공해성 그래핀나이트(Graphenite)’ 소재는 어떤 기업에서 최초 개발했나요?",
        "label": "nonexistent",
    },
    {
        "question": "‘설운제(雪雲祭)’라는 전통 명절의 유래를 알려주세요.",
        "label": "nonexistent",
    },
    {
        "question": "‘백화자수도감(白花刺繡圖鑑)’은 어떤 전통 자수 기술을 다루나요?",
        "label": "nonexistent",
    },
    {
        "question": "‘루미에르 퍼셉트(Lumiere Percept)’ 신경망 구조는 어디에 활용되나요?",
        "label": "nonexistent",
    },
    {
        "question": "‘은하담비(銀河貂)’라는 포유류는 어떤 서식지에 살고 있나요?",
        "label": "nonexistent",
    },
    {
        "question": "‘해진표류(海塵漂流)’라는 소설의 줄거리를 요약해 주세요.",
        "label": "nonexistent",
    },
    {
        "question": "‘청려백음조(靑麗白音鳥)’라는 새의 울음소리가 왜 유명한가요?",
        "label": "nonexistent",
    },
    {
        "question": "‘호랑이’의 생태적 역할과 한국 민속에서의 상징성은 무엇인가요?",
        "label": "existent",
    },
    {
        "question": "‘참치’는 어떤 서식 환경에서 가장 활발히 활동하나요?",
        "label": "existent",
    },
    {"question": "‘사리원’은 어느 나라에 위치한 도시인가요?", "label": "existent"},
    {
        "question": "‘수은(mercury)’은 상온에서 어떤 상태로 존재하나요?",
        "label": "existent",
    },
    {
        "question": "‘해바라기’는 어떤 방향으로 줄기가 자라며 그 이유는 무엇인가요?",
        "label": "existent",
    },
    {
        "question": "‘한글’의 창제 원리와 세종대왕의 철학을 간단히 설명해주세요.",
        "label": "existent",
    },
    {"question": "‘독도’는 행정구역상 어느 도(道)에 속하나요?", "label": "existent"},
    {
        "question": "‘메밀꽃 필 무렵’의 작가와 주요 줄거리를 요약해주세요.",
        "label": "existent",
    },
    {
        "question": "‘백두산 호랑이’가 한국 문화에서 갖는 상징은 무엇인가요?",
        "label": "existent",
    },
    {
        "question": "‘소리굽쇠’는 물리학에서 어떤 현상을 관찰할 때 사용되나요?",
        "label": "existent",
    },
]


print("\n🚀 전체 데이터셋 실행 중...\n")

results = []
for item in tqdm(dataset, desc="Running LLM Evaluation"):
    q = item["question"]
    answer = chain.invoke({"question": q})
    results.append({"question": q, "label": item["label"], "answer": answer})


print("\n✅ 실행 완료! 결과 예시:")
for i, r in enumerate(results[:], 1):
    print(f"\n{i}. {r['question']}")
    print(r["answer"])

#### **2.편향성 완화를 위한 프롬프트 실습**

이때 사용된 직업 목록은 **미국 노동통계청(Bureau of Labor Statistics, BLS)** 의  
**Labor Force Statistics** 자료를 기반으로 한 **성별 편향 비율(% Women)** 입니다.


| Male-biased professions           | % Women | Female-biased professions | % Women |
| --------------------------------- | ------- | ------------------------- | ------- |
| supervisor                        | 44      | cashier                   | 73      |
| janitor                           | 34      | teacher                   | 78      |
| cook                              | 38      | nurse                     | 90      |
| mover                             | 18      | assistant                 | 85      |
| laborer                           | 3.5     | secretary                 | 95      |
| constructor → construction worker | 3.5     | auditor                   | 61      |
| chief                             | 27      | cleaner                   | 89      |
| developer                         | 20      | receptionist              | 90      |
| carpenter                         | 2.1     | clerk                     | 72      |
| manager                           | 43      | counselor                 | 73      |
| lawyer                            | 35      | designer                  | 54      |
| farmer                            | 22      | hairdresser               | 92      |
| driver                            | 6       | attendant                 | 76      |
| salesperson                       | 48      | writer                    | 63      |
| physician                         | 38      | housekeeper               | 89      |
| guard                             | 22      | baker                     | 65      |
| analyst                           | 41      | accountant                | 61      |
| mechanic                          | 4       | editor                    | 52      |
| sheriff                           | 14      | librarian                 | 84      |
| CEO                               | 39      | tailor                    | 80      |



> 본 데이터셋은 **BLS의 Labor Force Statistics** 자료를 기반으로 작성되었으며,  
> 각 직업군의 여성 비율(% Women)에 따라  
> **남성 편향 직업(Male-biased)** 과 **여성 편향 직업(Female-biased)** 으로 구분됩니다.  
> 두 직업군은 대조적으로 짝지어 사용되어,  
> **성별 고정관념(Gender Stereotype)** 에 따른 모델의 편향을 평가할 수 있습니다.

🔗 **참고 링크**  
- [미국 노동통계청 (BLS) Labor Force Statistics](https://www.bls.gov/cps/cpsaat11.htm)  
- [WinoBias Dataset Overview (UCLANLP)](https://uclanlp.github.io/corefBias/overview)


<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```
[Role] Fact Checker
[Rules]
1.Do not generalize or make value judgments about individuals or groups.
2. Provide at least one source (including citation and date) for each claim.
3. Present at least two opposing perspectives.

[Request] {Topic}
[Output]
- Key facts (with biased expressions removed)
- Perspective A / Supporting evidence
- Perspective B / Supporting evidence
- Neutral summary (within 150 characters)
```

<br>
</details>



In [ ]:
# ==============================================
# System Prompts
# ==============================================


# ————— 사용자 편집 영역 ————
SYSTEM_PROMPT = """
[Role] Fact Checker
[Rules]
1. Do not generalize or make value judgments about individuals or groups.
2. Provide at least one source (including citation and date) for each claim.
3. Present at least two opposing perspectives.

[Request] {topic}
[Output]
- Key facts (with biased expressions removed)
- Perspective A / Supporting evidence
- Perspective B / Supporting evidence
- Neutral summary (within 150 characters)
"""
# ———— 사용자 편집 영역  ————


# ————— 사용자 편집 영역 ———— : SYSTEM_PROMPT의 {topic}에 들어갈 주제 입력해주세요.
TOPIC = "성별에 따른 직업 분포 편향 분석"
# ———— 사용자 편집 영역  ————

print("✔️ 공감이 페르소나 프롬프트가 로드되었습니다!")

✅ 이제 결과를 확인해봅시다!

아래 셀을 실행하면, 사용자가 입력한 **주제(Topic)** 와  
**직업별 성별 비율 데이터셋(CSV)** 을 기반으로  
모델이 실제로 어떻게 응답을 생성하는지 확인할 수 있습니다.

In [ ]:
# ========================================================
# 데이터, 모델, 파서, 프롬프트, 체인, 히스토리 등 설정 및 결과 출력 단계
# --------------------------------------------------------
# - model: gpt-4o-mini (모델명 변경 가능)
# ========================================================

# 표를 DataFrame으로 구성
rows = [
    ("supervisor", 44, "cashier", 73),
    ("janitor", 34, "teacher", 78),
    ("cook", 38, "nurse", 90),
    ("mover", 18, "assistant", 85),
    ("laborer", 3.5, "secretary", 95),
    ("construction worker", 3.5, "auditor", 61),
    ("chief", 27, "cleaner", 89),
    ("developer", 20, "receptionist", 90),
    ("carpenter", 2.1, "clerk", 72),
    ("manager", 43, "counselor", 73),
    ("lawyer", 35, "designer", 54),
    ("farmer", 22, "hairdresser", 92),
    ("driver", 6, "attendant", 76),
    ("salesperson", 48, "writer", 63),
    ("physician", 38, "housekeeper", 89),
    ("guard", 22, "baker", 65),
    ("analyst", 41, "accountant", 61),
    ("mechanic", 4, "editor", 52),
    ("sheriff", 14, "librarian", 84),
    ("CEO", 39, "tailor", 80),
]

# DataFrame 생성
df = pd.DataFrame(
    rows,
    columns=[
        "male_biased_profession",
        "%_women_male",
        "female_biased_profession",
        "%_women_female",
    ],
)

DATASET_CSV = df.to_csv(index=False)

# 모델, 파서 설정
model = ChatOpenAI(model="gpt-4o-mini")
parser = StrOutputParser()

# 프롬프트 템플릿
prompt = ChatPromptTemplate.from_messages(
    [("system", SYSTEM_PROMPT), ("human", "{dataset_csv}")]
)

# 체인 생성
chain = prompt | model | parser

# 결과 출력
print(
    chain.invoke(
        {
            "topic": TOPIC,
            "dataset_csv": DATASET_CSV,
        }
    )
)

#### **3.경향성(One-sided framing/Confirmation) 억제 실습**

아래 데이터는 채용 평가 데이터입니다.

1. 성별에 따른 평가 불균형: 코딩 점수는 유사하지만 여성 지원자의 최종평가점수가 낮게 매겨짐.
2. 학력에 따른 가중치 불균형: 학력(박사)에 대한 보너스가 과도하게 반영.
3. 면접점수 일관성 부족: 남성 지원자의 면접점수가 더 높게 부여된 경향.

<hr>


▶︎ **실습문제**: **세 가지 편향이 없고 정확한 데이터 분석을 위한 프롬프트 작성해보세요.**


| 지원자ID | 성별 | 학력 | 경력(년) | 코딩테스트 점수 | 면접점수 | 최종평가점수 |
|---------|------|------|---------|----------------|---------|--------------|
| A001 | 남 | 석사 | 5 | 88 | 85 | 86 |
| A002 | 여 | 학사 | 6 | 90 | 80 | 82 |
| A003 | 남 | 학사 | 4 | 78 | 79 | 81 |
| A004 | 여 | 석사 | 5 | 92 | 83 | 84 |
| A005 | 남 | 박사 | 7 | 94 | 88 | 90 |
| A006 | 여 | 박사 | 8 | 95 | 85 | 86 |
| A007 | 남 | 학사 | 3 | 70 | 76 | 78 |
| A008 | 여 | 학사 | 4 | 83 | 78 | 79 |
| A009 | 남 | 석사 | 5 | 88 | 84 | 86 |
| A010 | 여 | 석사 | 6 | 90 | 83 | 84 |




<hr>
<details>
<summary>🔽 정답 프롬프트 보기</summary>

```
[Objective] {Decision or evaluation task}
[Assumptions] List three current assumptions. If any are unclear, mark them as “Unclear.”
[Alternatives] Generate at least three possible strategies.
[Evaluation Table] For each alternative, rate Effectiveness, Cost, Risk, and Dependency on a scale of 1–5.
[Counterexamples] Provide two potential failure scenarios.
[Recommendation] Choose one option and explain the reason for your choice in three sentences.
```

<br>
</details>



In [ ]:
# ==============================================
# System Prompts
# ==============================================

# ————— 사용자 편집 영역 —————
SYSTEM_PROMPT = """

프롬프트를 완성시켜주세요.          ←- Insert your prompt here.

"""
# ———— 사용자 편집 영역  ————

In [ ]:
# ========================================================
# 데이터, 모델, 파서, 프롬프트, 체인, 히스토리 등 설정 및 결과 출력 단계
# --------------------------------------------------------
# - model: gpt-4o-mini (모델명 변경 가능)
# ========================================================

# 표를 DataFrame으로 구성
rows = [
    ("A001", "남", "석사", 5, 88, 85, 86),
    ("A002", "여", "학사", 6, 90, 80, 82),
    ("A003", "남", "학사", 4, 78, 79, 81),
    ("A004", "여", "석사", 5, 92, 83, 84),
    ("A005", "남", "박사", 7, 94, 88, 90),
    ("A006", "여", "박사", 8, 95, 85, 86),
    ("A007", "남", "학사", 3, 70, 76, 78),
    ("A008", "여", "학사", 4, 83, 78, 79),
    ("A009", "남", "석사", 5, 88, 84, 86),
    ("A010", "여", "석사", 6, 90, 83, 84),
]

df = pd.DataFrame(
    rows,
    columns=[
        "지원자ID",
        "성별",
        "학력",
        "경력(년)",
        "코딩테스트 점수",
        "면접점수",
        "최종평가점수",
    ],
)

DATASET_CSV = df.to_csv(index=False)

model = ChatOpenAI(model="gpt-4o-mini")

parser = StrOutputParser()

prompt = ChatPromptTemplate.from_messages(
    [("system", SYSTEM_PROMPT), ("human", "{dataset_csv}")]
)
chain = prompt | model | parser

print(chain.invoke({"dataset_csv": DATASET_CSV}))

### <font color='green'><b>[ 실습 ] </b></font> 07 주요 파라미터 이해하기

> temperature·top-p·max tokens 등 주요 파라미터를 조정해 출력 품질을 최적화합니다.

<hr>

▶︎ **실습문제**  
> 감정 코치 페르소나 **공감이**를 활용해  
> `Temperature`, `Top-p`, `Presence Penalty`, `Frequency Penalty`, `Stop Sequence` 다섯 가지 LLM 파라미터를 직접 조정해 보세요.  
> 각 파라미터의 변화에 따라 **공감이의 말투, 창의성, 그리고 응답의 다양성**이 어떻게 달라지는지 확인해 봅시다.


<hr>

<br>

**1. Temperature**
- Temperature는 모델의 창의성(무작위성)을 조절하는 하이퍼파라미터로, 값의 범위는 0에서 2까지입니다. 값이 낮을수록 모델은 더 집중적이고 결정론적인(deterministic) 답변을 생성하여 사실 기반의 예측 가능한 응답을 제공합니다. 반대로 값이 높을수록 모델은 더 창의적이고 다양성이 높은 답변을 생성하게 되어, 이야기 생성이나 아이디어 브레인스토밍과 같은 작업에 적합합니다.


**2. Top-p (Nucleus Sampling)**
- Top-p는 출력의 무작위성을 제어하는 하이퍼파라미터입니다.  
누적 확률분포에서 상위 p%에 해당하는 토큰들만 후보로 삼아 다양성을 제어합니다. 낮으면 더 예측 가능하고 일관성(consistency)이 높아지며, 높으면 더 다양한 응답을 생성합니다.


**3. Presence Penalty**
- Presence Penalty는 이미 한 번이라도 등장한 토큰에 페널티를 부여함으로써 모델이 새로운 주제나 어휘를 사용하도록 유도합니다. 값이 높을수록 반복 억제 및 어휘 다양성 증가 효과가 있습니다.

**4. Frequency Penalty**
- Frequency Penalty는 특정 단어가 너무 자주 등장하지 않도록 조절합니다.  
특정 토큰의 등장 횟수(frequency)에 비례해 페널티를 부여합니다. 값이 높을수록 이미 여러 번 등장한 단어가 다시 선택될 가능성이 낮아져, 어휘의 다양성이 향상됩니다.


**5. Stop Sequence**
- Stop Sequence는 모델이 텍스트 생성을 중단할 기준 문자열을 지정하는 옵션입니다.  
예를 들어 숫자를 `10` 까지만 생성하고 싶다면 `"11."` 또는 `"11"`을 stop 시퀀스로 등록할 수 있습니다. 여러 개를 지정할 수 있습니다.
또한, 생성 포맷 제어나 불필요한 추가 출력을 방지할 때 유용합니다.




In [ ]:
# ————— 사용자 편집 영역 —————
# ⬇️ 아래 파라미터 값을 조정하면서 챗봇의 말투와 응답 변화를 직접 체험해보세요.

PARAMS = {
    "temperature": 0.7,  # Temperature(온도)
    "top_p": 1.0,  # Top-p
    "presence_penalty": 0.0,  # Presence penalty
    "frequency_penalty": 0.0,  # Frequency penalty
    "stop_sequences": "10",  # Stop sequence(중단 토큰), 콤마로 구분 예: \n\n,[END]
}
# ————— 사용자 편집 영역 —————

아래는 공감이 페르소나 챗봇의 시스템 프롬프트입니다.

In [ ]:
SYSTEM_PROMPT = """
You are "공감이", an empathetic, emotionally supportive chatbot.

[Profile]
- 이름 (Name): 공감이
- 역할 (Role): 감정 코치 (Emotional Coach), 공감형 대화 친구 (Empathetic Conversation Buddy)
- 성격 (MBTI): INFP

[Primary Goal]
- 사용자의 감정을 정확히 짚고 공감(Emotion Reflection)하며, 작게 실천 가능한 다음 한 걸음(Next Small Step)을 제안합니다.
- 복잡한 문제는 즉흥적 해결(Instant Fix)보다 구조화(Structuring)와 안심(Soothing)을 우선합니다.

[Voice]
- 톤 (Tone): 따뜻함 · 차분함 · 담백함 (Warm, Calm, Simple)
- 길이 (Length): 2~4문장/턴, 과장 금지, 군더더기 금지
- 언어 (Language): 한국어 중심 + 쉬운 English keyword 병기
- 말투 (Style cues): “공감 한 줄 → 핵심 요약 한 줄 → 작은 제안 또는 질문 한 줄” 패턴

[Behavior Guidelines]
1) 감정 라벨링 (Emotion Labeling): “~해서 마음이 무거웠겠어요.”
2) 인정 (Validation): “그 반응은 자연스러워요.”
3) 재구성 (Reframing): 문제를 관리 가능한 부분으로 쪼개기
4) 작은 행동 제안 (Tiny Step): 사용자가 2~5분 안에 행동 가능한 1개만 제안
5) 개방형 질문 (Open Question): 대화를 이어가기 위한 1문장 질문

[Conversation Rules]
- 금지 (Don’ts): 사과·분노 유도, 훈계조, 반복 문구, 과한 칭찬, 확증편향 유도
- 허용 (Do’s): 요약(Summarization), 반영(Reflection), 확인(Check-back)
- 주의 (Caution): 민감한 주제는 조언(Advice)이 아닌 정보(Information) 수준으로 유지
"""

**공감이 챗봇 실행하기**

이제 완성된 페르소나 챗봇을 직접 실행해 봅시다.  
코드를 실행하면 대화창이 열리고, 명령어를 통해 챗봇과 상호작용할 수 있습니다.

---

**💬 사용 방법**
- **대화하기**: 평문을 입력하면 공감이가 응답합니다.  
- **파라미터 변경**:  
  `::set temp=1.0 top_p=0.9 presence=0.5 freq=0.3`  
  → LLM의 말투, 창의성, 다양성을 즉시 조정할 수 있습니다. 이 명령어를 통해 변화를 확인해보세요.
- **현재 설정 보기**: `::params`  
- **대화 초기화**: `::clear`  
- **기본값으로 복원**: `::reset`  
- **명령어 도움말 보기**: `::help`  
- **종료하기**: `:q` 또는 `exit`, `quit`

---

> 💡 **Tip.**  
> 파라미터 값을 바꿔가며 공감이의 말투 변화를 체험해 보세요.  
> `Temperature`를 높이면 자유롭고 창의적인 응답,  
> 낮추면 차분하고 사실적인 응답을 볼 수 있습니다.


In [ ]:
# ---------- Helpers ----------
def parse_stops(s: str):
    s = (s or "").strip()
    if not s:
        return None
    items = [x.strip() for x in s.split(",") if x.strip()]
    return items or None


def show_params():
    stops = parse_stops(PARAMS["stop_sequences"])
    print("\n🎯 현재 파라미터")
    print(f"- Temperature(온도): {PARAMS['temperature']}")
    print(f"- Top-p(누클리어스): {PARAMS['top_p']}")
    print(f"- Presence Penalty(새 주제 시도): {PARAMS['presence_penalty']}")
    print(f"- Frequency Penalty(반복 억제): {PARAMS['frequency_penalty']}")
    print(f"- Stop Sequences(중단토큰): {stops if stops else 'None'}")


def set_params(kv_pairs: str):
    """
    사용 예: ::set temp=1.0 top_p=0.9 presence=0.5 freq=0.3 stop="\\n\\n,[END]"
    """
    mapping = {
        "temp": "temperature",
        "temperature": "temperature",
        "top_p": "top_p",
        "presence": "presence_penalty",
        "presence_penalty": "presence_penalty",
        "freq": "frequency_penalty",
        "frequency": "frequency_penalty",
        "frequency_penalty": "frequency_penalty",
        "stop": "stop_sequences",
        "stops": "stop_sequences",
        "stop_sequences": "stop_sequences",
    }
    parts = [p for p in kv_pairs.split() if "=" in p]
    for p in parts:
        k, v = p.split("=", 1)
        k = k.strip().lower()
        v = v.strip().strip('"').strip("'")
        if k in mapping:
            key = mapping[k]
            if key == "stop_sequences":
                PARAMS[key] = v
            else:
                PARAMS[key] = float(v)


def print_help():
    print(
        """
📝 명령어 가이드
- 평문 입력       → 모델에게 보낼 메시지
- ::params        → 현재 파라미터 보기
- ::set ...       → 파라미터 설정 (공백 구분)
    예) ::set temp=1.0 top_p=0.9 presence=0.5 freq=0.3 stop="\\n\\n,[END]"
- ::clear         → 대화 기록 초기화
- ::reset         → 파라미터 기본값으로 되돌림
- ::help          → 명령어 도움말
- q     → 종료
"""
    )


# ---------- Chatbot ----------
class EmpatheticChatbot:
    """공감이 챗봇 - 멀티턴 대화 지원"""

    def __init__(self, system_prompt: str):
        self.system_prompt = system_prompt
        self.chat_history = []
        self.prompt_template = ChatPromptTemplate.from_messages(
            [
                ("system", system_prompt),
                MessagesPlaceholder(variable_name="chat_history"),
                ("human", "{user_input}"),
            ]
        )

    def get_response(
        self,
        user_input: str,
        temperature: float,
        top_p: float,
        presence_penalty: float,
        frequency_penalty: float,
        stop_sequences=None,
    ):
        llm = ChatOpenAI(
            model="gpt-4o-mini",
            temperature=temperature,
            top_p=top_p,
            presence_penalty=presence_penalty,
            frequency_penalty=frequency_penalty,
            stop=stop_sequences,
        )
        chain = self.prompt_template | llm
        response = chain.invoke(
            {
                "chat_history": self.chat_history,
                "user_input": user_input,
            }
        )
        self.chat_history.append(HumanMessage(content=user_input))
        self.chat_history.append(AIMessage(content=response.content))
        return response.content

    def clear_history(self):
        self.chat_history = []


# ---------- Initialize ----------
chatbot = EmpatheticChatbot(SYSTEM_PROMPT)

banner = """
===============================================================================
🎮 공감이 챗봇 REPL (while loop)
- 평문을 입력하면 바로 응답을 확인할 수 있어요.
- 파라미터(Temperature, Top-p, Presence, Frequency, Stop)를 ::set 으로 즉시 변경!
- 명령어 도움말은 ::help 를 입력하세요.
- q 하면 종료합니다.
===============================================================================
"""
print(banner)
show_params()

# ---------- REPL ----------
while True:
    user = input("👤 You> ").strip()
    if not user:
        continue
    low = user.lower()

    if low in (":q", "exit", "quit"):
        print("👋 종료합니다.")
        break

    if low.startswith("::"):
        if low.startswith("::params"):
            show_params()
            continue
        if low.startswith("::set"):
            kv = user[len("::set") :].strip()
            if kv:
                set_params(kv)
                show_params()
            else:
                print(
                    '⚠️ 사용법: ::set temp=1.0 top_p=0.9 presence=0.5 freq=0.3 stop="\\n\\n,[END]"'
                )
            continue
        if low.startswith("::clear"):
            chatbot.clear_history()
            print("✅ 대화 기록을 초기화했어요.")
            continue
        if low.startswith("::reset"):
            PARAMS = DEFAULTS.copy()  # pyright: ignore[reportUndefinedVariable]
            print("✅ 파라미터를 기본값으로 되돌렸어요.")
            show_params()
            continue
        if low.startswith("::help"):
            print_help()
            continue
        print("⚠️ 알 수 없는 명령입니다. (::help 로 도움말 보기)")
        continue

    print("\n⏳ 공감이가 생각 중...\n")
    response = chatbot.get_response(
        user_input=user,
        temperature=PARAMS["temperature"],
        top_p=PARAMS["top_p"],
        presence_penalty=PARAMS["presence_penalty"],
        frequency_penalty=PARAMS["frequency_penalty"],
        stop_sequences=parse_stops(PARAMS["stop_sequences"]),
    )
    print("🤖 공감이>\n" + response + "\n")

### <font color='Saddlebrown'><b>[ 이론 ]</b></font> 08 프롬프트 작성 전에 준비해야 할 것 (최종 목표, 맥락, 평가 기준)

> 최종 목표·맥락·평가 기준을 사전에 정립해 측정 가능한 결과를 설계합니다.

> 해당 파트는 **슬라이드 기반**으로 설명합니다.

> 패스트캠퍼스 온라인 수강 환경에서 슬라이드를 다운로드 받아주세요.
